# 03 - Renewable Generation Forecasting
## Leakage-Safe Feature Engineering, Baseline Evaluation, and ML Models

This notebook trains day-ahead hourly solar forecasting models using strictly historical information (lag >= 24h), solar geometry clear-sky proxies, and evaluates them against Persistence and Seasonal Naive baselines.


In [1]:
import sys
sys.path.insert(0, '..')

from src.preprocessing import get_clean_data
from src.features import build_forecasting_features, verify_feature_leakage
from src.forecasting import train_and_evaluate_models
import pandas as pd

clean_df = get_clean_data()
print("Running anti-leakage verification test...")
verify_feature_leakage(clean_df)


[2026-10-25 13:11:09] [INFO] [src.preprocessing]: Loading cached clean data from /home/asus/Projects/Renewable-Energy-Intelligence/data/processed/DK_1_clean.parquet


[2026-10-25 13:11:09] [INFO] [src.features]: Leakage verification test PASSED. Features are strictly non-anticipative.


Running anti-leakage verification test...


True

### 1. Train Models & Evaluate on Chronological Holdout Set (2020)

In [2]:
preds, metrics, leaderboard = train_and_evaluate_models(clean_df, force_retrain=False)
print("=== Model Leaderboard (Sorted by MAE) ===")
display(leaderboard)


[2026-10-25 13:11:09] [INFO] [src.forecasting]: Chronological split sizes: Train=34,729 (2015-01-14 to 2018-12-31), Val=8,760 (2019-01-01 to 2019-12-31), Test=6,576 (2020-01-01 to 2020-09-30)


/home/asus/anaconda3/envs/renewable-ei/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/home/asus/anaconda3/envs/renewable-ei/lib/python3.11/site-packages/lightgbm/basic.py:668: UserWarning: Usage of np.ndarray subset (sliced data) is not recommended due to it will double the peak memory cost in LightGBM.
  _log_warning(


[2026-10-25 13:11:11] [INFO] [src.forecasting]: Model training and evaluation successfully completed!


[2026-10-25 13:11:11] [INFO] [src.forecasting]: 
                   model               type  mae_all  rmse_all  nmae_all_pct  mae_daylight  skill_score  train_time_sec  inference_time_ms  size_kb
0      Persistence (24h)           Baseline   40.360    86.850          5.55        71.176       0.0000           0.000               0.00      0.0
1       Ridge Regression          Linear ML   42.246    77.034          5.81        72.873      -0.0467           0.009               0.34      0.7
2          Random Forest        Ensemble ML   43.880    82.502          6.04        77.841      -0.0872           0.960              22.92  13569.1
3               LightGBM  Gradient Boosting   44.171    82.493          6.08        76.586      -0.0944           0.181               6.45    231.7
4  Seasonal Naive (168h)           Baseline   52.482   104.562          7.22        92.815      -0.3003           0.000               0.00      0.0


=== Model Leaderboard (Sorted by MAE) ===


,model,type,mae_all,rmse_all,nmae_all_pct,mae_daylight,skill_score,train_time_sec,inference_time_ms,size_kb
0,Persistence (24h),Baseline,40.360,86.850,5.55,71.176,0.0000,0.000,0.00,0.0
1,Ridge Regression,Linear ML,42.246,77.034,5.81,72.873,-0.0467,0.009,0.34,0.7
2,Random Forest,Ensemble ML,43.880,82.502,6.04,77.841,-0.0872,0.960,22.92,13569.1
3,LightGBM,Gradient Boosting,44.171,82.493,6.08,76.586,-0.0944,0.181,6.45,231.7
4,Seasonal Naive (168h),Baseline,52.482,104.562,7.22,92.815,-0.3003,0.000,0.00,0.0


### 2. Conformal Prediction Intervals & Empirical Coverage

In [3]:
print("90% Conformal Prediction Interval Metrics:")
print(metrics.get("Conformal_90_Coverage"))
print("80% Conformal Prediction Interval Metrics:")
print(metrics.get("Conformal_80_Coverage"))


90% Conformal Prediction Interval Metrics:
{'empirical_coverage_pct': 70.35, 'mean_interval_width_mw': 83.658}
80% Conformal Prediction Interval Metrics:
{'empirical_coverage_pct': 60.07, 'mean_interval_width_mw': 46.814}
